# 第 1 周练习 —— 商业洞察讲解器

## 练习目标（理念）

做一个小工具：给定「分析某科技公司商业模式」的问题，用大语言模型给出结构化讲解。

本 notebook **对比两条后端**：

- **GPT-4o-mini**：经 **OpenRouter**（OpenAI 兼容 `base_url`）调用托管模型
- **Llama 3.2**：经本机 **Ollama** 原生 `/api/chat`（HTTP + `requests`）

重点练习：提示词设计（system 定专家角色 + user 列分析提纲），并肉眼对比托管模型与本地模型的结构清晰度。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | `SYSTEM_PROMPT` + `question` |
| OpenAI 兼容网关 | OpenRouter：`base_url=https://openrouter.ai/api/v1` |
| Ollama 本地模型 | `POST {OLLAMA_URL}/api/chat`，`stream: False` |

## 怎么跑

1. 准备 `.env`：至少有 `OPENAI_KEY`（给 OpenRouter）；可选 `OLLAMA_URL`（默认 `http://localhost:11434`）
2. 本地要跑 Llama：先启动 Ollama 并 `ollama pull llama3.2`
3. 从上到下运行；可把 `question` 换成别的公司再对比两格输出


In [ ]:
# ========== 导入：环境、HTTP、OpenAI SDK、笔记本展示 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如密钥与 Ollama 地址
import os
# 导入标准库 requests：用 HTTP 调用本地 Ollama 的 /api/chat
import requests
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：这里会指向 OpenRouter 的兼容端点
from openai import OpenAI
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 环境 + 模型名 + OpenRouter 客户端 ==========

# 加载 .env：把 OPENAI_KEY、OLLAMA_URL 等读入进程环境
load_dotenv()

# OpenAI / OpenRouter 侧要用的模型 id（字符串须与网关支持的名字一致）
MODEL_GPT = "gpt-4o-mini"
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = "llama3.2"
# OpenRouter 的 OpenAI 兼容 API 根地址
openrouter_url = "https://openrouter.ai/api/v1"

# Ollama 基址：优先读环境变量；没有则默认本机 11434
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")

# 创建 OpenAI 客户端，但 base_url 指向 OpenRouter；密钥来自 OPENAI_KEY（不是 OPENAI_API_KEY）
client = OpenAI(base_url=openrouter_url, api_key=os.getenv("OPENAI_KEY"))


In [ ]:
# ========== 提示词：system 定专家角色，user 放具体分析提纲 ==========

# SYSTEM_PROMPT 保持英文：这是发给模型的指令，改译会改变回答风格/行为
SYSTEM_PROMPT = """
You are an expert in startup and SaaS business models.

Provide clear and structured explanations that help developers
and entrepreneurs understand how technology companies operate.
"""

# question：示例用户问题（分析 Stripe）；发给模型的内容保持英文
question = """
Analyze the business model of Stripe.

Provide:
1. Short summary
2. Revenue model
3. Target customers
4. Competitive advantage
"""


In [ ]:
def ask_gpt(question: str) -> str:
    """用 OpenRouter 上的 GPT 模型做一次非流式 Chat Completions，返回纯文本回复。"""

    # chat.completions.create：标准 Chat Completions；messages 含 system + user
    response = client.chat.completions.create(
        # 使用上面定义的 MODEL_GPT（gpt-4o-mini）
        model=MODEL_GPT,
        messages=[
            # system：商业模式专家人设
            {"role": "system", "content": SYSTEM_PROMPT},
            # user：具体公司分析问题
            {"role": "user", "content": question},
        ],
    )

    # 非流式：整段回复在 choices[0].message.content
    return response.choices[0].message.content


In [ ]:
# ========== 调用 GPT 并在笔记本里展示 ==========

# 把示例 question 发给 GPT，拿到 Markdown 风格文本
gpt_result = ask_gpt(question)

# 先显示分区标题，再渲染模型全文（标题字符串保持原样，便于对照输出）
display(Markdown("## GPT-4o-mini Response"))
display(Markdown(gpt_result))


In [ ]:
def ask_llama(question: str) -> str:
    """通过 Ollama 原生 HTTP /api/chat 查询本地 Llama，返回 message.content 文本。"""

    # POST 到 {OLLAMA_URL}/api/chat；json 里 stream=False 表示一次返回完整 JSON
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            # 本地模型名须与 ollama list 一致
            "model": MODEL_LLAMA,
            "messages": [
                # 与 GPT 共用同一套 system / user，方便公平对比
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            # 关闭流式：响应体是单个 JSON，而不是 NDJSON 行流
            "stream": False
        },
    )

    # 解析 JSON；Ollama 非流式时正文在 data["message"]["content"]
    data = response.json()
    return data["message"]["content"]


In [ ]:
# ========== 调用本地 Llama 并在笔记本里展示 ==========

# 同一 question 再问一遍本地模型，便于并排对比结构与详略
llama_result = ask_llama(question)

# 分区标题保持英文原样；正文用 Markdown 渲染
display(Markdown("## Llama 3.2 Response"))
display(Markdown(llama_result))


In [ ]:
# ========== 观察记录：作者对两模型输出的主观对比（打印字符串保持英文原样）==========

# 先打印小节，方便在 stdout 里定位这段笔记
print("Observation:")

# 多行字符串：记录「GPT 更结构化 / Llama 更短但仍抓住要点」——可运行展示文案不翻译
print("""
GPT produced a more structured explanation and clearer breakdown of the business model.

Llama generated a shorter response but still captured the main revenue model
and core target users.

This shows how prompt structure can guide models to produce useful business insights.
""")
